# testConditionalGANPytorch1  
Andrew E. Davidson aedavids@ucsc.edu 8/29/24  

Copyright (c) 2020-2023, Regents of the University of California All rights reserved.   https://polyformproject.org/licenses/noncommercial/1.0.0

AIM: create a simple GAN that is easy to test our basic framework

generate y = x^2

ref: 
- chapter 6. in Generative Advisarial Networks with Python  
    this does not work. Keras/tensor flow version issues?  
    re-write example using pytorch

- [pytorch doc](https://pytorch.org/docs/stable/index.html)

In [1]:
import ipynbname
import matplotlib.pyplot as plt

from numpy import hstack
from numpy import zeros
from numpy import ones
from numpy.random import rand
from numpy.random import randn
import os

# by default keras use tensorflow as backend
import torch
print(f'torch.__version__: {torch.__version__}')

from torch import nn
torch.manual_seed(0) # Set for testing purposes, please do not change!

notebookName = ipynbname.name()
notebookPath = ipynbname.path()
notebookDir = os.path.dirname(notebookPath)

outDir = f'{notebookDir}/{notebookName}.out'
imgOut = f'{outDir}/img'
print(f'imgOut:\n{imgOut}')

torch.__version__: 2.5.1.post102
imgOut:
/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/gan/testConditionalGAN_pytorch_1.out/img


## Create models

In [12]:
class ParabolaDiscriminator( nn.Module ):
    def __init__( self, inputSize : int ):
        '''
            inputSize, the length of the generated vectors
        '''
        super().__init__()

        # keras
        # model = Sequential()
    	# model.add(Dense(25, activation='relu', kernel_initializer='he_uniform', input_dim=n_inputs))
    	# model.add(Dense(1, activation='sigmoid'))
        
        self.model = nn.Sequential(
            nn.Linear(inputSize, 25),
            nn.ReLU(),
            nn.Linear(25, 1),
            nn.Sigmoid()
        )
        
        # Initialize weights
        nn.init.kaiming_uniform_(self.model[0].weight, nonlinearity='relu')
        nn.init.zeros_(self.model[0].bias)
        nn.init.kaiming_uniform_(self.model[2].weight, nonlinearity='sigmoid')
        nn.init.zeros_(self.model[2].bias)

    def forward( self, X : torch.Tensor ):
        ret = self.model( X )
        return ret

def testParabolaDiscriminator() :
    inputSize = 2
    discriminator = ParabolaDiscriminator( inputSize )
    numTest = 2
    testInput = torch.randn( numTest, inputSize )
    predictions = discriminator( testInput )
    print(f'\npredictions :\n{predictions}')

testParabolaDiscriminator()


predictions :
tensor([[0.4279],
        [0.3088]], grad_fn=<SigmoidBackward0>)


In [3]:
class ParabolaGenerator( nn.Module ):
    def __init__( self, latentDim : int, nOutputs : int = 2 ) :
        '''
        arguments:
            latentDim: the length of the noise input vector

            nOutputs : the length of the output layers vector. Default = 2
        '''
        super().__init__()


        # keras
        # model.add(Dense(15, activation='relu', kernel_initializer='he_uniform', input_dim=latent_dim))
        # model.add(Dense(n_outputs, activation='linear'))
        
        self.model = nn.Sequential(
            nn.Linear(latentDim, 15),
            nn.ReLU(),
            nn.Linear(15, nOutputs),
        )
        # Initialize weights
        nn.init.kaiming_uniform_(self.model[0].weight, nonlinearity='relu')
        nn.init.zeros_(self.model[0].bias)
        nn.init.kaiming_uniform_(self.model[2].weight, nonlinearity='linear')
        nn.init.zeros_(self.model[2].bias)

    def forward( self, noise : torch.Tensor ):
        '''
            noise should be the value returned by generateLatentPoints()
        '''
        ret = self.model( noise )
        return ret

## Data Utilities
function to generate real and fake tensors

In [4]:
def generateRealSamples( n : int ) -> tuple[torch.Tensor, torch.Tensor] :
    '''
    generate n real parabola samples with class labels

    Returns 2 Tensor
        X, y i.e. (realSamples, realLabels)

        y = 1, ie real
    '''
    # generate inputs in range [-0.5, 0.5]
    X1 = rand(n) - 0.5
    
    # generate outputs X^2
    X2 = X1 * X1
    
    # stack arrays
    X1 = X1.reshape(n, 1)
    X2 = X2.reshape(n, 1)
    X = hstack((X1, X2))
    
    # generate class labels
    y = ones((n, 1))
    
    realSamples = torch.Tensor( X )
    realLabels = torch.Tensor( y)
    
    return (realSamples, realLabels)

def testGenerateRealSamples() :
    X, y = generateRealSamples( n = 5 ) 
    print( f'X.shape: {X.shape} rank : {len(X.shape)} num elements : {X.numel()}' )
    print( X )

    print( f'\ny.shape: {y.shape} rank : {len(y.shape)} num elements : {y.numel()}' )
    print ( y) 

testGenerateRealSamples()

X.shape: torch.Size([5, 2]) rank : 2 num elements : 10
tensor([[0.4297, 0.1847],
        [0.2129, 0.0453],
        [0.3336, 0.1113],
        [0.3401, 0.1157],
        [0.2596, 0.0674]])

y.shape: torch.Size([5, 1]) rank : 2 num elements : 5
tensor([[1.],
        [1.],
        [1.],
        [1.],
        [1.]])


In [5]:
def generateLatentPoints(latentDimensions : int = 5, 
                         n : int = 100) -> torch.Tensor :
    '''
    generate points in latent space as input for the generator

    latentDimensions:
        the number of dimensions for the generator's input vector

    n the number of vectors to generate

    returns a tensor
    '''
    # generate points in the latent space
    xInput = randn(latentDimensions * n)
    
    # reshape into a batch of inputs for the network
    xInput = xInput.reshape(n, latentDimensions)
    
    ret = torch.Tensor( xInput )
    
    return ret

def testGenerateLatentPoints():
    tglp = generateLatentPoints(latentDimensions=5, n=3 )
    print( tglp )
    print( tglp.shape )

testGenerateLatentPoints()

tensor([[-1.7915,  0.5997, -1.2517, -0.3869, -0.2359],
        [ 0.4471, -1.1372, -0.1969,  0.7215, -1.4471],
        [-1.6224,  0.7975,  0.9601,  0.2402, -0.4800]])
torch.Size([3, 5])


In [6]:
def generateFakeSamples(generator : ParabolaGenerator, 
                        latentDimensions : int,
                        n : int) -> tuple[torch.Tensor, torch.Tensor]:
    '''
    use the generator to generate n fake examples, with class labels

    returns (fakeSamples, fakeLabels)
        labels will be zeros.
    '''
    # generate points in latent space
    noiseVector = generateLatentPoints(latentDimensions, n)

    # forward() is the old way of doing things
    # detach 
    #fakeSamples = generator(noiseVector).detach()

    # Set the model to evaluation mode
    generator.eval()

    # Disable gradient calculation for inference
    with torch.no_grad():
        fakeSamples = generator(noiseVector)
    
    # create class labels  
    y = zeros((n, 1))
    fakeLabels = torch.Tensor( y )
    
    return fakeSamples, fakeLabels

def testGenerateFakeSamples():
    latentDimensions = 5
    numOutputs = 2
    

    gen = ParabolaGenerator( latentDimensions, numOutputs )

    numSamples = 3
    fakeSamples, fakeLabels = generateFakeSamples( gen, latentDimensions, numSamples)

    print( f'\nfakeSamples.shape: {fakeSamples.shape} rank : {len(fakeSamples.shape)} num elements : {fakeSamples.numel()}' )
    print( fakeSamples )

    print( f'\nfakeLabels.shape: {fakeLabels.shape} rank : {len(fakeLabels.shape)} num elements : {fakeLabels.numel()}' )
    print ( fakeLabels ) 


testGenerateFakeSamples()


fakeSamples.shape: torch.Size([3, 2]) rank : 2 num elements : 6
tensor([[ 0.3156, -0.0033],
        [-0.3120, -1.4183],
        [-0.6465, -1.6380]])

fakeLabels.shape: torch.Size([3, 1]) rank : 2 num elements : 3
tensor([[0.],
        [0.],
        [0.]])


NameError: name 'aedwip' is not defined

## Train Model

In [ ]:
# create a discrimanator
discriminatorModel = ParabolaDiscriminator( inputSize=2 )
# define the cost function
discriminatorCriterion = nn.BCELoss()

# use stochastic gradient decent
# keras
# 	discrModel.compile(loss='binary_crossentropy', optimizer='adam', 
# metrics=['accuracy'])

learningRate = 0.01
discriminatorOptimizer = torch.optim.adam( discriminatorModel.parameters, 
                                          lr=learningRate )

# define the number of training loops
numEpochs = aedwip
for t in range( numEpochs ) :

    # forward propagation
    # get a prediction
    yHat = discriminatorModel( X )
    discriminatorLoss = discriminatorCriterion( yHat, y)

    # backward propagation
    discriminatorOptimizer.zero_grad()
    discriminatorLoss.backward()

    # update the parameters
    discriminatorOptimizer.step()
    